In [1]:

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

from pathlib import Path

import pandas as pd
import numpy as np


# ==========================================
# 2. PROJECT CONFIGURATION
# ==========================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Make sure data/processed exists."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR = PROCESSED_DIR / "contextual"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ==========================================
# 3. CONTEXT CONFIGURATION
# ==========================================

SHORT_CONTEXT_SIZE = 2
EXTENDED_CONTEXT_SIZE = 5

SORT_COLUMNS = [
    "match_id",
    "conversation_id",
    "chat_time",
    "Id"
]

print("Context construction configuration loaded!")
print("Project root:", PROJECT_ROOT)
print("Input directory:", PROCESSED_DIR)
print("Output directory:", OUTPUT_DIR)

Context construction configuration loaded!
Project root: C:\Users\celes\Documents\GitHub\197-miniproject
Input directory: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed
Output directory: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed\contextual


In [2]:
# ==========================================
# 1. LOAD PROCESSED DATASETS
# ==========================================

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
valid_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")


# ==========================================
# 2. ORGANIZE DATASET PARTITIONS
# ==========================================

datasets = {
    "Train": train_df,
    "Validation": valid_df,
    "Test": test_df
}


# ==========================================
# 3. DISPLAY DATASET INFORMATION
# ==========================================

for name, data in datasets.items():
    print(f"\n{name} Dataset")
    print("-" * 40)
    print("Rows:", len(data))
    print("Unique matches:", data["match_id"].nunique())
    print("Unique conversations:", data["conversation_id"].nunique())
    print("Missing utterances:", data["utterance"].isna().sum())


# ==========================================
# 4. VALIDATE REQUIRED COLUMNS
# ==========================================

required_columns = SORT_COLUMNS + [
    "utterance",
    "intent_class",
    "toxicity_binary"
]

for name, data in datasets.items():
    missing_columns = set(required_columns) - set(data.columns)

    if missing_columns:
        raise ValueError(
            f"{name} dataset is missing columns: {missing_columns}"
        )

print("\nAll datasets loaded and required columns are present!")


Train Dataset
----------------------------------------
Rows: 25297
Unique matches: 1341
Unique conversations: 7667
Missing utterances: 0

Validation Dataset
----------------------------------------
Rows: 5165
Unique matches: 288
Unique conversations: 1537
Missing utterances: 0

Test Dataset
----------------------------------------
Rows: 5425
Unique matches: 288
Unique conversations: 1697
Missing utterances: 0

All datasets loaded and required columns are present!


In [3]:
# ==========================================
# 1. DEFINE CONTEXT CONSTRUCTION FUNCTION
# ==========================================

def add_context_column(partition, context_size):
    """
    Add a context column containing up to `context_size`
    preceding messages from the same match and conversation.
    """

    # ==========================================
    # 2. SORT MESSAGES CHRONOLOGICALLY
    # ==========================================

    partition = partition.sort_values(
        SORT_COLUMNS,
        kind="stable"
    ).copy()

    contextual_groups = []

    # ==========================================
    # 3. GROUP MESSAGES BY MATCH AND CONVERSATION
    # ==========================================

    for _, group in partition.groupby(
        ["match_id", "conversation_id"],
        sort=False
    ):
        group = group.copy()
        messages = group["utterance"].astype(str).tolist()

        contexts = []

        # ==========================================
        # 4. CONSTRUCT CONTEXT WINDOWS
        # ==========================================

        for i, target in enumerate(messages):

            # Get only the preceding messages
            previous_messages = messages[max(0, i - context_size):i]

            # Format preceding messages
            context_parts = [
                f"[PREVIOUS] {message}"
                for message in previous_messages
            ]

            # Append the target message
            context_parts.append(f"[TARGET] {target}")

            contexts.append("\n".join(context_parts))

        # Add generated contexts to the conversation
        group[f"context_{context_size}"] = contexts
        contextual_groups.append(group)

    # ==========================================
    # 5. COMBINE CONTEXTUALIZED CONVERSATIONS
    # ==========================================

    return pd.concat(contextual_groups).reset_index(drop=True)


print("Context-building function defined successfully!")

Context-building function defined successfully!


In [4]:
# ==========================================
# 1. SELECT A SAMPLE CONVERSATION
# ==========================================

first_group = next(
    iter(
        train_df.groupby(
            ["match_id", "conversation_id"],
            sort=False
        )
    )
)

sample_conversation = first_group[1]


# ==========================================
# 2. GENERATE SAMPLE CONTEXT WINDOWS
# ==========================================

sample_context_2 = add_context_column(
    sample_conversation,
    SHORT_CONTEXT_SIZE
)

sample_context_5 = add_context_column(
    sample_conversation,
    EXTENDED_CONTEXT_SIZE
)


# ==========================================
# 3. DISPLAY CONTEXT WINDOW PREVIEW
# ==========================================

for i in range(min(3, len(sample_context_2))):

    print(f"\nMESSAGE {i + 1}")
    print("=" * 50)

    print("SHORT CONTEXT (2 preceding messages):")
    print(sample_context_2.iloc[i]["context_2"])

    print("\nEXTENDED CONTEXT (5 preceding messages):")
    print(sample_context_5.iloc[i]["context_5"])

    print("-" * 50)


MESSAGE 1
SHORT CONTEXT (2 preceding messages):
[TARGET] force it

EXTENDED CONTEXT (5 preceding messages):
[TARGET] force it
--------------------------------------------------

MESSAGE 2
SHORT CONTEXT (2 preceding messages):
[PREVIOUS] force it
[TARGET] space created [SEPA] hah

EXTENDED CONTEXT (5 preceding messages):
[PREVIOUS] force it
[TARGET] space created [SEPA] hah
--------------------------------------------------

MESSAGE 3
SHORT CONTEXT (2 preceding messages):
[PREVIOUS] force it
[PREVIOUS] space created [SEPA] hah
[TARGET] ez 500

EXTENDED CONTEXT (5 preceding messages):
[PREVIOUS] force it
[PREVIOUS] space created [SEPA] hah
[TARGET] ez 500
--------------------------------------------------


In [5]:
# ==========================================
# 1. GENERATE SHORT-CONTEXT DATASETS
# ==========================================

train_context_2 = add_context_column(
    train_df, SHORT_CONTEXT_SIZE
)

valid_context_2 = add_context_column(
    valid_df, SHORT_CONTEXT_SIZE
)

test_context_2 = add_context_column(
    test_df, SHORT_CONTEXT_SIZE
)


# ==========================================
# 2. GENERATE EXTENDED-CONTEXT DATASETS
# ==========================================

train_context_5 = add_context_column(
    train_df, EXTENDED_CONTEXT_SIZE
)

valid_context_5 = add_context_column(
    valid_df, EXTENDED_CONTEXT_SIZE
)

test_context_5 = add_context_column(
    test_df, EXTENDED_CONTEXT_SIZE
)


# ==========================================
# 3. DISPLAY CONTEXT GENERATION SUMMARY
# ==========================================

print("Context generation complete!")

for name, data in {
    "Train - Short": train_context_2,
    "Validation - Short": valid_context_2,
    "Test - Short": test_context_2,
    "Train - Extended": train_context_5,
    "Validation - Extended": valid_context_5,
    "Test - Extended": test_context_5
}.items():

    print(f"{name}: {len(data):,} messages")

Context generation complete!
Train - Short: 25,297 messages
Validation - Short: 5,165 messages
Test - Short: 5,425 messages
Train - Extended: 25,297 messages
Validation - Extended: 5,165 messages
Test - Extended: 5,425 messages


In [6]:
# ==========================================
# 1. VALIDATE ROW COUNTS
# ==========================================

assert len(train_context_2) == len(train_df)
assert len(valid_context_2) == len(valid_df)
assert len(test_context_2) == len(test_df)

assert len(train_context_5) == len(train_df)
assert len(valid_context_5) == len(valid_df)
assert len(test_context_5) == len(test_df)


# ==========================================
# 2. VALIDATE MESSAGE IDS AND LABELS
# ==========================================

dataset_pairs = [
    (train_df, train_context_2),
    (valid_df, valid_context_2),
    (test_df, test_context_2),
    (train_df, train_context_5),
    (valid_df, valid_context_5),
    (test_df, test_context_5)
]

for original, contextual in dataset_pairs:

    assert original["Id"].tolist() == contextual["Id"].tolist()

    assert (
        original["intent_class"].tolist()
        == contextual["intent_class"].tolist()
    )

    assert (
        original["toxicity_binary"].tolist()
        == contextual["toxicity_binary"].tolist()
    )


# ==========================================
# 3. VALIDATE MATCH SEPARATION
# ==========================================

train_matches = set(train_df["match_id"])
valid_matches = set(valid_df["match_id"])
test_matches = set(test_df["match_id"])

assert train_matches.isdisjoint(valid_matches)
assert train_matches.isdisjoint(test_matches)
assert valid_matches.isdisjoint(test_matches)


# ==========================================
# 4. VALIDATE TARGET MESSAGE ALIGNMENT
# ==========================================

contextual_datasets = [
    train_context_2, valid_context_2, test_context_2,
    train_context_5, valid_context_5, test_context_5
]

for contextual in contextual_datasets:

    context_column = (
        "context_2"
        if "context_2" in contextual.columns
        else "context_5"
    )

    for _, row in contextual.iterrows():

        expected_target = f"[TARGET] {row['utterance']}"
        actual_target = row[context_column].split("\n")[-1]

        assert actual_target == expected_target


# ==========================================
# 5. DISPLAY VALIDATION RESULTS
# ==========================================

print("All integrity checks passed!")
print("Row counts: PASS")
print("Message IDs and labels: PASS")
print("Match separation: PASS")
print("Target-message alignment: PASS")

All integrity checks passed!
Row counts: PASS
Message IDs and labels: PASS
Match separation: PASS
Target-message alignment: PASS


In [7]:
# ==========================================
# 1. SAVE SHORT-CONTEXT DATASETS
# ==========================================

train_context_2.to_csv(
    OUTPUT_DIR / "train_context_2.csv",
    index=False
)

valid_context_2.to_csv(
    OUTPUT_DIR / "validation_context_2.csv",
    index=False
)

test_context_2.to_csv(
    OUTPUT_DIR / "test_context_2.csv",
    index=False
)


# ==========================================
# 2. SAVE EXTENDED-CONTEXT DATASETS
# ==========================================

train_context_5.to_csv(
    OUTPUT_DIR / "train_context_5.csv",
    index=False
)

valid_context_5.to_csv(
    OUTPUT_DIR / "validation_context_5.csv",
    index=False
)

test_context_5.to_csv(
    OUTPUT_DIR / "test_context_5.csv",
    index=False
)


# ==========================================
# 3. DISPLAY SAVING SUMMARY
# ==========================================

print("All six contextual datasets saved successfully!")
print("Output directory:", OUTPUT_DIR)

All six contextual datasets saved successfully!
Output directory: C:\Users\celes\Documents\GitHub\197-miniproject\data\processed\contextual


In [8]:
# ==========================================
# 1. IDENTIFY SAVED DATASET FILES
# ==========================================

saved_files = sorted(OUTPUT_DIR.glob("*.csv"))

assert len(saved_files) == 6, (
    f"Expected 6 files, but found {len(saved_files)}"
)


# ==========================================
# 2. VERIFY DATASET ROW COUNTS
# ==========================================

for file in saved_files:

    check_df = pd.read_csv(file)

    print(f"{file.name}: {len(check_df):,} rows")


# ==========================================
# 3. DISPLAY VERIFICATION RESULT
# ==========================================

print("\nAll six contextual datasets verified!")

test_context_2.csv: 5,425 rows
test_context_5.csv: 5,425 rows
train_context_2.csv: 25,297 rows
train_context_5.csv: 25,297 rows
validation_context_2.csv: 5,165 rows
validation_context_5.csv: 5,165 rows

All six contextual datasets verified!
